In [ ]:
import sys

!uv pip install --python "{sys.executable}" -q -U torch torchvision transformers accelerate torchcodec scikit-learn pandas numpy matplotlib tqdm requests ipywidgets


In [ ]:
import shutil
import subprocess
import sys

if shutil.which("ffmpeg") is None:
    print("ffmpeg not found on PATH; torchcodec needs it to decode video files.")
    if sys.platform.startswith("linux"):
        print("Attempting `apt-get install -y ffmpeg` (requires root)...")
        subprocess.run(["apt-get", "update", "-qq"], check=False)
        subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=False)
    else:
        print("Please install ffmpeg manually, e.g. `conda install ffmpeg` or your OS package manager.")
else:
    print("ffmpeg found:", shutil.which("ffmpeg"))


In [ ]:
import random
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import torch
from IPython.display import display
from tqdm.auto import tqdm

try:
    from torchcodec.decoders import VideoDecoder
except Exception as exc:
    raise ImportError(
        "torchcodec failed to import, most likely because the FFmpeg shared libraries "
        "are missing. Install FFmpeg (see the cell above, or `conda install ffmpeg`) "
        "and restart the kernel."
    ) from exc

from transformers import (
    AutoConfig,
    AutoModelForVideoClassification,
    AutoVideoProcessor,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score


In [ ]:
MODEL_NAME = "facebook/vjepa2-vitl-fpc64-256"

NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR / "dataset"
OUTPUT_DIR = NOTEBOOK_DIR / "checkpoints" / "vjepa2-chicken-action"

DOWNLOAD_CHICKENDET = True
EXTRACT_CHICKENDET = False

SEED = 42
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.8, 0.1, 0.1

BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 8
NUM_EPOCHS = 20
LEARNING_RATE = 1e-4
FREEZE_ENCODER = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Dataset directory: {DATA_DIR}")


In [ ]:
ZENODO_FILES = {
    "ChickenAct.zip": "https://zenodo.org/records/20672799/files/ChickenAct.zip?download=1",
}


def download_file(url: str, dest: Path, chunk_size: int = 1 << 20) -> None:
    if dest.exists() and dest.stat().st_size > 0:
        print(f"Skipping download, already present: {dest}")
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp_dest = dest.with_suffix(dest.suffix + ".part")
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(tmp_dest, "wb") as f, tqdm(
            total=total, unit="B", unit_scale=True, unit_divisor=1024, desc=dest.name
        ) as bar:
            for chunk in r.iter_content(chunk_size=chunk_size):
                f.write(chunk)
                bar.update(len(chunk))
    tmp_dest.rename(dest)


for filename, url in ZENODO_FILES.items():
    if filename == "ChickenDet.zip" and not DOWNLOAD_CHICKENDET:
        continue
    download_file(url, DATA_DIR / filename)


In [ ]:
!unzip -q -o ./dataset/ChickenAct.zip -d ./dataset/ChickenAct


In [ ]:
VIDEO_EXTENSIONS = {".mp4"}
SPLIT_FOLDER_NAMES = {"train", "training", "val", "valid", "validation", "test", "testing"}

FARM_PATTERNS = [
    re.compile(r"^[Ff](\d+)"),
    re.compile(r"^(\d+)"),
    re.compile(r"farm[_-]?(\d+)", re.IGNORECASE),
]


def extract_farm_id(filename: str) -> str:
    stem = Path(filename).stem
    for pattern in FARM_PATTERNS:
        match = pattern.match(stem)
        if match:
            return match.group(1)
    return "unknown"


def infer_class_label(video_path: Path, dataset_root: Path) -> str:
    parent = video_path.parent
    if parent.name.lower() in SPLIT_FOLDER_NAMES and parent.parent != dataset_root.parent:
        return parent.parent.name
    return parent.name


def build_manifest(dataset_root: Path) -> pd.DataFrame:
    rows = []
    for path in dataset_root.rglob("*"):
        if path.suffix.lower() not in VIDEO_EXTENSIONS:
            continue
        rows.append(
            {
                "path": str(path),
                "class": infer_class_label(path, dataset_root),
                "farm": extract_farm_id(path.name),
            }
        )
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(
            f"No video files found under {dataset_root}. Check the extracted folder layout "
            "and update VIDEO_EXTENSIONS / infer_class_label if it differs from expectations."
        )
    return df


manifest_df = build_manifest(DATA_DIR / "ChickenAct")
print(
    f"Found {len(manifest_df)} clips across {manifest_df['class'].nunique()} classes and "
    f"{manifest_df['farm'].nunique()} distinct farm ids."
)

unknown_farm = (manifest_df["farm"] == "unknown").sum()
if unknown_farm:
    examples = manifest_df.loc[manifest_df["farm"] == "unknown", "path"].head(5).tolist()
    print(f"WARNING: {unknown_farm} files did not match a farm-number prefix pattern.")
    print(f"Examples: {examples}")
    print("Inspect these filenames and adjust FARM_PATTERNS above if the naming convention differs.")

manifest_df.head()


In [ ]:
SEQUENTIAL_BLUE = "#2a78d6"
CATEGORICAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]

class_counts = manifest_df["class"].value_counts().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(class_counts.index, class_counts.values, color=SEQUENTIAL_BLUE, width=0.6)
ax.set_ylabel("Number of clips")
ax.set_title("ChickenAct clips per behavior class")
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="x", rotation=45)
for label in ax.get_xticklabels():
    label.set_ha("right")
plt.tight_layout()
plt.show()

farm_counts = manifest_df["farm"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(farm_counts.index.astype(str), farm_counts.values, color=SEQUENTIAL_BLUE, width=0.6)
ax.set_ylabel("Number of clips")
ax.set_xlabel("Farm id")
ax.set_title("ChickenAct clips per source farm")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
def allocate_split_counts(n: int, rng: random.Random) -> tuple[int, int, int]:
    if n <= 0:
        return 0, 0, 0
    if n == 1:
        return 1, 0, 0
    if n == 2:
        return (1, 1, 0) if rng.random() < 0.5 else (1, 0, 1)

    raw = [n * TRAIN_FRAC, n * VAL_FRAC, n * TEST_FRAC]
    counts = [int(x) for x in raw]
    remainder = n - sum(counts)
    order = sorted(range(3), key=lambda i: raw[i] - counts[i], reverse=True)
    for i in range(remainder):
        counts[order[i % 3]] += 1

    n_train, n_val, n_test = counts
    if n_val == 0:
        n_val, n_train = 1, n_train - 1
    if n_test == 0:
        n_test = 1
        if n_train >= n_val:
            n_train -= 1
        else:
            n_val -= 1

    assert n_train + n_val + n_test == n and min(n_train, n_val, n_test) >= 0
    return n_train, n_val, n_test


rng = random.Random(SEED)
train_idx, val_idx, test_idx = [], [], []

for _, group in manifest_df.groupby(["class", "farm"], sort=False):
    idx = group.index.tolist()
    rng.shuffle(idx)
    n_train, n_val, n_test = allocate_split_counts(len(idx), rng)
    train_idx += idx[:n_train]
    val_idx += idx[n_train : n_train + n_val]
    test_idx += idx[n_train + n_val : n_train + n_val + n_test]

train_df = manifest_df.loc[train_idx].reset_index(drop=True)
val_df = manifest_df.loc[val_idx].reset_index(drop=True)
test_df = manifest_df.loc[test_idx].reset_index(drop=True)

total = len(manifest_df)
print(f"train: {len(train_df)} ({len(train_df) / total:.1%})")
print(f"val:   {len(val_df)} ({len(val_df) / total:.1%})")
print(f"test:  {len(test_df)} ({len(test_df) / total:.1%})")


In [ ]:
split_class_pct = pd.DataFrame(
    {
        "train": train_df["class"].value_counts(normalize=True),
        "val": val_df["class"].value_counts(normalize=True),
        "test": test_df["class"].value_counts(normalize=True),
    }
).fillna(0).sort_index()
display(split_class_pct.style.format("{:.1%}").background_gradient(cmap="Blues", axis=1))

print("Distinct farms per split:")
print(
    {
        "train": train_df["farm"].nunique(),
        "val": val_df["farm"].nunique(),
        "test": test_df["farm"].nunique(),
    }
)
print("\nDistinct farms represented per class (val):")
print(val_df.groupby("class")["farm"].nunique())
print("\nDistinct farms represented per class (test):")
print(test_df.groupby("class")["farm"].nunique())

fig, ax = plt.subplots(figsize=(4, 4))
splits, sizes = ["train", "val", "test"], [len(train_df), len(val_df), len(test_df)]
ax.bar(splits, sizes, color=CATEGORICAL[:3], width=0.5)
ax.set_ylabel("Number of clips")
ax.set_title("Split sizes")
ax.spines[["top", "right"]].set_visible(False)
for i, v in enumerate(sizes):
    ax.text(i, v, f"{v}\n({v / total:.0%})", ha="center", va="bottom")
plt.tight_layout()
plt.show()


In [ ]:
SPLITS_DIR = DATA_DIR / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
train_df.to_csv(SPLITS_DIR / "train.csv", index=False)
val_df.to_csv(SPLITS_DIR / "val.csv", index=False)
test_df.to_csv(SPLITS_DIR / "test.csv", index=False)
print(f"Saved split manifests to {SPLITS_DIR}")

classes = sorted(manifest_df["class"].unique())
label2id = {c: i for i, c in enumerate(classes)}
id2label = {i: c for c, i in label2id.items()}
num_classes = len(classes)
print(f"{num_classes} classes: {classes}")

for split_df in (train_df, val_df, test_df):
    split_df["label_id"] = split_df["class"].map(label2id)


In [ ]:
NUM_FRAMES = AutoConfig.from_pretrained(MODEL_NAME).frames_per_clip
print(f"Model expects {NUM_FRAMES} frames per clip")


def sample_frame_indices(total_frames: int, num_frames: int) -> np.ndarray:
    if total_frames <= 0:
        raise ValueError("Video has no frames.")
    if total_frames >= num_frames:
        return np.round(np.linspace(0, total_frames - 1, num_frames)).astype(int)
    indices = np.arange(total_frames)
    pad = np.full(num_frames - total_frames, total_frames - 1)
    return np.concatenate([indices, pad])


class ChickenActionDataset(torch.utils.data.Dataset):
    def __init__(self, df: pd.DataFrame, num_frames: int):
        self.df = df.reset_index(drop=True)
        self.num_frames = num_frames

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        decoder = VideoDecoder(row["path"])
        total_frames = decoder.metadata.num_frames
        frame_idx = sample_frame_indices(total_frames, self.num_frames)
        frames = decoder.get_frames_at(indices=frame_idx.tolist()).data
        return frames, int(row["label_id"])


In [ ]:
processor = AutoVideoProcessor.from_pretrained(MODEL_NAME)

train_ds = ChickenActionDataset(train_df, NUM_FRAMES)
val_ds = ChickenActionDataset(val_df, NUM_FRAMES)
test_ds = ChickenActionDataset(test_df, NUM_FRAMES)


def collate_fn(batch):
    videos, labels = zip(*batch)
    inputs = processor(list(videos), return_tensors="pt")
    inputs["labels"] = torch.tensor(labels, dtype=torch.long)
    return inputs


sample_loader = torch.utils.data.DataLoader(train_ds, batch_size=2, collate_fn=collate_fn)
batch = next(iter(sample_loader))
print({k: v.shape for k, v in batch.items()})


In [ ]:
model = AutoModelForVideoClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
    attn_implementation="sdpa",
)

if FREEZE_ENCODER:
    for name, param in model.named_parameters():
        if not any(key in name.lower() for key in ("classifier", "pooler")):
            param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total_params:,} ({trainable / total_params:.1%})")


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }


training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=10,
    report_to=[],
    remove_unused_columns=False,
    bf16=torch.cuda.is_available(),
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)


In [ ]:
trainer.train()


In [ ]:
test_metrics = trainer.evaluate(test_ds, metric_key_prefix="test")
print(test_metrics)

predictions = trainer.predict(test_ds)
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=-1)

print(classification_report(y_true, y_pred, target_names=classes))


In [ ]:
cm = confusion_matrix(y_true, y_pred, normalize="true")

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(num_classes))
ax.set_xticklabels(classes, rotation=45, ha="right")
ax.set_yticks(range(num_classes))
ax.set_yticklabels(classes)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Test set confusion matrix (row-normalized)")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


In [ ]:
FINAL_DIR = OUTPUT_DIR / "final"
trainer.save_model(str(FINAL_DIR))
processor.save_pretrained(str(FINAL_DIR))
print(f"Saved fine-tuned model and processor to {FINAL_DIR}")
